# Macro Regime Engine — walkthrough

Runs the whole pipeline from a notebook: pull data, fit the hidden Markov model, label the states,
set a scenario override, and render the HTML report inline.

Two ways to run it:

* **Demo mode** (default below) uses a synthetic 3-regime history and needs no FRED key, so every cell works immediately.
* **Real mode**: set `FRED_API_KEY` in your environment (or pass `api_key=...`) and use `MacroRegime()`.

Run the notebook from the repository root (or add it to `sys.path`, as the first cell does).

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "regime.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from regime import MacroRegime

DEMO = True                      # set False once FRED_API_KEY is exported
mr = MacroRegime(demo=DEMO)      # real mode: MacroRegime() or MacroRegime(api_key="...")
mr

## 1. Data
`refresh()` pulls the raw FRED series (or the synthetic ones in demo mode), builds the monthly headline
panel and caches everything as Parquet. `panel()` is what a human reads; `features()` is what the HMM sees.

In [ ]:
panel = mr.refresh()
panel.tail()

In [ ]:
mr.features().tail()

## 2. Fit
`refit()` is the same job the monthly scheduler runs: it builds the training set from the **actual** cache only
(overrides can never reach it), runs EM with several restarts, orders the states from most bearish-looking
to most bullish-looking, and saves the model.

In [ ]:
engine = mr.refit(refresh=False)
mr.states().round(2)

## 3. Label the states (human in the loop)
Look at the per-state means above. `label()` with no argument accepts the heuristic suggestion; pass a
mapping to set them yourself. Labels are stored against the `fit_id`, so a refit that reshuffles states
shows as *provisional* until you confirm again.

In [ ]:
mr.label()                     # or: mr.label({0: "bearish", 1: "transition", 2: "bullish"})

## 4. Score and report
`summary()` gives the actual vs. scenario probabilities; `report()` renders the full HTML report inline.

In [ ]:
mr.summary().round(3)

In [ ]:
mr.report()

## 5. Scenario overrides
Overrides are in **headline** units (PMI level, spread in pp, Core PCE YoY %, Fed Funds %). They apply to
the latest month only; the stationary transforms are recomputed on top of the actual history.

In [ ]:
mr.override(ism_pmi=44, yield_curve=-0.3)
mr.summary().round(3)

In [ ]:
report = mr.report()
report.save(ROOT / "cache" / "regime_report.html")   # standalone file; inline_js=True embeds plotly.js for offline use
report

In [ ]:
mr.clear_overrides()

## 5b. Choosing indicators
The catalogue has 15 indicators (`docs/indicators.md`); 7 are active by default. `evaluate_indicators()`
runs a walk-forward ablation of each candidate against the active set and flags the ones that add signal.
On real data this is the step to run before changing `MACRO_REGIME_INDICATORS` or `MacroRegime(indicators=[...])`.

In [ ]:
print("active:", mr.indicators)
mr.evaluate_indicators(candidates=["payrolls", "cpi", "consumer_sentiment"], refit_every=24, min_train=180).round(3)

## 6. Walk-forward backtest
Out-of-sample check: refit every `refit_every` months on data strictly before each cut, then filter forward.
In demo mode the true regime path is known, so we can measure accuracy.

In [ ]:
bt = mr.backtest(refit_every=12, min_train=120)
print("out-of-sample accuracy vs. true regimes:", round(bt.accuracy(mr.true_states), 3) if DEMO else "n/a")
bt.summary().round(2)

In [ ]:
from reporting.figures import history_figure
labels = mr.labeler.resolve(mr.engine).labels
history_figure(bt.posteriors, labels, title="Out-of-sample filtered probabilities")

## Next steps
* Real data: `export FRED_API_KEY=...`, set `DEMO = False`, rerun from the top.
* ISM PMI is not on FRED since 2016: either set `MACRO_REGIME_PMI_SERIES` to a proxy or keep prints in
  `cache/manual/NAPM.csv` (`date,value`).
* Dashboard: `python -m dashboard.app`. Monthly refit: `python scheduler.py --cron`.